# Real-data customer segmentation

Maintained analysis of the versioned Kaggle input. Retrieve the documented local CSV first. This is descriptive batch fitting, not supervised evaluation. Models and records stay local.


In [ ]:
from pathlib import Path
from uuid import uuid4
import pandas as pd
project = Path.cwd()
if project.name == 'notebooks':
    project = project.parent
if not (project / 'pyproject.toml').exists():
    raise RuntimeError('Start Jupyter from the segmentation project or its notebooks folder.')
from customer_segments.train import train
input_path = project / 'data/marketing_campaign.csv'
if not input_path.exists():
    raise FileNotFoundError('Retrieve the source CSV as documented in data/README.md, or use quickstart.ipynb.')
run = project / 'artifacts' / ('notebook-real-' + uuid4().hex[:8])
metrics = train(input_path, project / 'configs/default.json', run)
print('Rows:', metrics['rows'], 'Selected K:', metrics['selected_k'])
print('In-sample silhouette:', metrics['silhouette'])
print('Initialization checks:', metrics['initialization_stability'])


## Aggregate profiles

Descriptions refer to K-Means only. Observed income means skip missing entries; counts are included. GMM IDs are independent.


In [ ]:
pd.read_csv(run / 'cluster_profiles.csv')


## Diagnostics

Silhouette compares K values on this cohort. Heatmap colors standardize across segment means; annotations retain original units. Neither demonstrates campaign lift.


In [ ]:
from customer_segments.report import report
report(run, run / 'report')
from IPython.display import display, Image
display(Image(filename=str(run / 'report/model-selection.png')))
display(Image(filename=str(run / 'report/segment-profiles.png')))


## Saved-model reload check

This reassigns fitted input rows for consistency; it is not a holdout evaluation. No individual customer records are displayed.


In [ ]:
from customer_segments.predict import predict
assigned = predict(run / 'segmentation.joblib', input_path, run / 'reloaded-assignments.csv')
print('Assigned rows:', len(assigned))
print('GMM boundary rows:', int(assigned.gmm_boundary.sum()))


See `MODEL_CARD.md`, `data/README.md`, and `docs/results.md` for interpretation, source attribution, and limitations.
